# Performance Report

This notebook backfills yearly `_full` IBKR Flex XML archives, refreshes the current-year `_latest` XML, rebuilds `performance_history.feather`, previews the latest statement and history tail, and generates the dated performance report with historical `Y<year>` rows appended.


In [ ]:
from dataclasses import asdict
from datetime import date, datetime
import os
from pathlib import Path
import sys
import time
import xml.etree.ElementTree as ET

import pandas as pd
from IPython.display import Markdown, display

In [ ]:
display(
    pd.Series(
        {
            "python_executable": sys.executable,
            "python_version": sys.version.splitlines()[0],
            "conda_env": os.environ.get("CONDA_DEFAULT_ENV", ""),
            "cwd": str(Path.cwd()),
        },
        name="notebook_runtime",
    )
)

In [ ]:



def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "market_helper").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing 'market_helper'.")


def read_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def local_name(tag: str) -> str:
    return tag.rsplit("}", 1)[-1]


def first_statement(root: ET.Element) -> ET.Element:
    for element in root.iter():
        if local_name(element.tag) == "FlexStatement":
            return element
    raise RuntimeError("Downloaded payload did not include a FlexStatement node.")


def archive_records_to_frame(records) -> pd.DataFrame:
    rows = []
    for record in records:
        payload = asdict(record)
        payload["target_path"] = str(payload["target_path"])
        payload["source_file"] = str(payload["source_file"]) if payload["source_file"] is not None else ""
        payload["xml_from_date"] = payload["xml_from_date"].isoformat() if payload["xml_from_date"] is not None else ""
        payload["xml_to_date"] = payload["xml_to_date"].isoformat() if payload["xml_to_date"] is not None else ""
        payload["error_message"] = payload.get("error_message") or ""
        rows.append(payload)
    return pd.DataFrame(rows)


def run_step(label: str, func, /, *args, **kwargs):
    started_at = datetime.now()
    started_perf = time.perf_counter()
    print(f"{label}:start at={started_at.isoformat(timespec='seconds')}")
    try:
        result = func(*args, **kwargs)
    except Exception as exc:
        elapsed_seconds = time.perf_counter() - started_perf
        print(
            f"{label}:error after={elapsed_seconds:.2f}s type={type(exc).__name__} message={exc}"
        )
        raise
    elapsed_seconds = time.perf_counter() - started_perf
    print(f"{label}:done after={elapsed_seconds:.2f}s")
    return result


print("setup:A locate project root")
PROJECT_ROOT = find_project_root(Path.cwd())
print("setup:B project root", PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("setup:C sys.path updated")

print("setup:D import performance analytics")
from market_helper.domain.portfolio_monitor.services.performance_analytics import (
    annualized_return,
    annualized_vol,
    drawdown_plot_frame,
    performance_plot_frame,
    sharpe_ratio,
)
print("setup:E import performance history")
from market_helper.domain.portfolio_monitor.services.performance_history import load_performance_history
print("setup:F import workflow functions")
from market_helper.workflows.generate_report import (
    backfill_ibkr_flex_full_years,
    generate_ibkr_flex_performance_report,
    rebuild_ibkr_flex_performance_history,
    refresh_current_year_latest_flex_xml,
)

CONFIG_ENV_PATH = PROJECT_ROOT / "configs" / "portfolio_monitor" / "local.env"
OUTPUT_DIR = PROJECT_ROOT / "data" / "artifacts" / "portfolio_monitor" / "flex"
RAW_XML_DIR = OUTPUT_DIR / "raw"
HISTORY_PATH = OUTPUT_DIR / "performance_history.feather"
RAW_XML_DIR.mkdir(parents=True, exist_ok=True)

ENV_CONFIG = read_env_file(CONFIG_ENV_PATH)

# Optional notebook-local override if you do not want to store the query id in local.env.
FLEX_QUERY_ID_OVERRIDE = ""

FLEX_QUERY_ID = (
    FLEX_QUERY_ID_OVERRIDE.strip()
    or os.environ.get("IBKR_FLEX_QUERY_ID", "").strip()
    or ENV_CONFIG.get("IBKR_FLEX_QUERY_ID", "").strip()
)
FLEX_TOKEN = (
    os.environ.get("IBKR_FLEX_TOKEN", "").strip()
    or ENV_CONFIG.get("IBKR_FLEX_TOKEN", "").strip()
)

ARCHIVE_START_YEAR = 2020
BACKFILL_END_YEAR = 2025
OVERWRITE_FULL_ARCHIVE = False

# IBKR often needs longer than the library defaults for large or historical Flex ranges.
FLEX_POLL_INTERVAL_SECONDS = 10.0
FLEX_MAX_ATTEMPTS = 3

config_series = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "config_env_path": str(CONFIG_ENV_PATH),
        "raw_xml_dir": str(RAW_XML_DIR),
        "history_path": str(HISTORY_PATH),
        "has_token": bool(FLEX_TOKEN),
        "has_query_id": bool(FLEX_QUERY_ID),
        "archive_start_year": ARCHIVE_START_YEAR,
        "backfill_end_year": BACKFILL_END_YEAR,
        "flex_poll_interval_seconds": FLEX_POLL_INTERVAL_SECONDS,
        "flex_max_attempts": FLEX_MAX_ATTEMPTS,
    },
    name="flex_notebook_config",
)
print("setup:G display config")
display(config_series)


## Historical Full-Year XML Backfill

This cell downloads or promotes yearly `ibkr_flex_<YYYY>_full.xml` archives for every year from `ARCHIVE_START_YEAR` to `BACKFILL_END_YEAR`.


In [ ]:
if not FLEX_QUERY_ID:
    raise RuntimeError(
        "Set IBKR_FLEX_QUERY_ID in configs/portfolio_monitor/local.env, your shell env, or FLEX_QUERY_ID_OVERRIDE in the setup cell."
    )
if not FLEX_TOKEN:
    raise RuntimeError("Set IBKR_FLEX_TOKEN in configs/portfolio_monitor/local.env or your shell env.")

full_year_records = run_step(
    "backfill:full-years",
    backfill_ibkr_flex_full_years,
    output_dir=OUTPUT_DIR,
    query_id=FLEX_QUERY_ID,
    token=FLEX_TOKEN,
    start_year=ARCHIVE_START_YEAR,
    end_year=BACKFILL_END_YEAR,
    overwrite_existing=OVERWRITE_FULL_ARCHIVE,
    poll_interval_seconds=FLEX_POLL_INTERVAL_SECONDS,
    max_attempts=FLEX_MAX_ATTEMPTS,
)

full_year_archive_df = (
    archive_records_to_frame(full_year_records)
    .sort_values(["year", "kind"], kind="stable")
    .reset_index(drop=True)
)
full_year_archive_df["target_exists"] = full_year_archive_df["target_path"].map(lambda raw: Path(raw).exists())

display(
    full_year_archive_df.loc[
        :,
        [
            "year",
            "kind",
            "status",
            "target_exists",
            "target_path",
            "source_file",
            "xml_from_date",
            "xml_to_date",
            "error_message",
        ],
    ]
)

display(
    pd.DataFrame(
        {
            "full_xml_file": [path.name for path in sorted(RAW_XML_DIR.glob("ibkr_flex_*_full.xml"))],
        }
    )
)


## Current-Year Latest XML Refresh

This cell refreshes the rolling `ibkr_flex_<current_year>_latest.xml` archive for the current year YTD range.


In [ ]:
print(
    f"refresh:params poll_interval={FLEX_POLL_INTERVAL_SECONDS}s max_attempts={FLEX_MAX_ATTEMPTS}"
)
latest_record = run_step(
    "refresh:latest-xml",
    refresh_current_year_latest_flex_xml,
    output_dir=OUTPUT_DIR,
    query_id=FLEX_QUERY_ID,
    token=FLEX_TOKEN,
    poll_interval_seconds=FLEX_POLL_INTERVAL_SECONDS,
    max_attempts=FLEX_MAX_ATTEMPTS,
)
print(
    f"refresh:result status={latest_record.status} xml_from={latest_record.xml_from_date} xml_to={latest_record.xml_to_date} path={latest_record.target_path}"
)
display(archive_records_to_frame([latest_record]))


## Rebuild Feather History

This cell rebuilds `performance_history.feather` from the archived `_full` files plus the current `_latest` file.


In [ ]:
history_path = run_step(
    "history:rebuild-feather",
    rebuild_ibkr_flex_performance_history,
    output_dir=OUTPUT_DIR,
    extra_xml_paths=[latest_record.target_path],
)
display(
    pd.DataFrame(
        [
            {
                "history_path": str(history_path),
                "exists": history_path.exists(),
                "size_bytes": history_path.stat().st_size if history_path.exists() else None,
            }
        ]
    )
)


In [ ]:
latest_xml_text = latest_record.target_path.read_text(encoding="utf-8")
latest_root = ET.fromstring(latest_xml_text)
latest_statement = first_statement(latest_root)
latest_lines = latest_xml_text.splitlines()
latest_preview = "\n".join(latest_lines[:60])

latest_summary_df = pd.DataFrame(
    [
        {
            "saved_xml_path": str(latest_record.target_path),
            "statement_from_date": latest_statement.get("fromDate", ""),
            "statement_to_date": latest_statement.get("toDate", ""),
            "statement_period": latest_statement.get("period", ""),
            "when_generated": latest_statement.get("whenGenerated", ""),
            "account_id": latest_statement.get("accountId", ""),
            "preview_lines": min(60, len(latest_lines)),
        }
    ]
)

display(latest_summary_df)
display(Markdown("```xml\n" + latest_preview + "\n```"))


In [ ]:
history_df = run_step("history:load-feather", load_performance_history, history_path)
display(history_df.loc[:, [
    "date",
    "nav_close_usd",
    "nav_close_sgd",
    "cash_flow_usd",
    "cash_flow_sgd",
    "twr_return_usd",
    "twr_return_sgd",
    "is_final",
    "source_kind",
]].tail(12))


In [ ]:
report_path = run_step(
    "report:generate-csv",
    generate_ibkr_flex_performance_report,
    flex_xml_path=latest_record.target_path,
    output_dir=OUTPUT_DIR,
)
display(
    pd.DataFrame(
        [
            {
                "report_path": str(report_path),
                "exists": report_path.exists(),
                "size_bytes": report_path.stat().st_size if report_path.exists() else None,
            }
        ]
    )
)


In [ ]:
report_df = pd.read_csv(report_path)
display(report_df)


In [ ]:
display(report_df.isna().sum().rename("missing_values"))


In [ ]:
analytics_summary = pd.DataFrame(
    [
        {
            "currency": currency,
            "annualized_return": annualized_return(history_df, currency),
            "annualized_vol": annualized_vol(history_df, currency),
            "sharpe_ratio": sharpe_ratio(history_df, currency),
        }
        for currency in ("USD", "SGD")
    ]
)
display(analytics_summary)
display(performance_plot_frame(history_df, "USD").tail())
display(drawdown_plot_frame(history_df, "USD").tail())
